**Running Classifier on the SSL and handcrafted acoustic features and comparing with human ambiguity on emotions**

A simple logistic-regression classifier is trained on each feature set separately:
the 36 MATLAB acoustic features and each of the 13 layers of WavLM and UniSpeech-SAT
(27 feature sets), using leave-one-actor-out evaluation (24 folds)

As suggested in the feedback from project supervisor, we wont use more advanced models for the sake of gettitng better at emotion detection as extensive research as already been done on it.

Instead our objective is how features from SSL and handcrafted acoustic features compare to humans in terms of which emotions they struggle to identify most.

In [ ]:
# importing data preprocessor and model libraries

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import pandas as pd

In [ ]:
# defining the raw database contaiing the the y labels (emotions)

meta  = pd.read_csv("results/metadata.csv")
folds = pd.read_csv("results/folds.csv")
acou  = pd.read_csv("results/acoustic_features.csv")

# target: the intended emotion
y = meta["emotion"].values

In [ ]:
# SSL layer features dataframefrom extract_ssl.py, shape (1440, 13, 768)          
wavlm  = np.load("results/wavlm_layers.npy")                                 
unisat = np.load("results/unispeechsat_layers.npy")    

In [ ]:
# converting the features data into easier matrix form for classifier training
systems = [("acoustic", -1, acou.drop(columns=["filename"]).values)]        
for layer in range(13):                                                     
    systems.append(("wavlm", layer, wavlm[:, layer, :]))                    
    systems.append(("unispeechsat", layer, unisat[:, layer, :])) 
              


In [ ]:
rows, fold_scores = [], []

for system, layer, X in systems:                 # 27 feature sets
    print("Running", system, "layer", layer)
    for fold in range(1, 25):                    # leave-one-actor-out
        train = (folds["fold"] != fold).values
        test  = (folds["fold"] == fold).values

        model = make_pipeline(StandardScaler(),
                              LogisticRegression(C=0.01, max_iter=2000, class_weight="balanced"))
        model.fit(X[train], y[train])            # train on 23 actors
        probs, pred = model.predict_proba(X[test]), model.predict(X[test])

        # score for this held-out actor
        fold_scores.append({"system": system, "layer": layer, "fold": fold,
                            "macro_f1": f1_score(y[test], pred, average="macro"),
                            "accuracy": accuracy_score(y[test], pred)})

        # per-clip prediction and confidence in the intended emotion
        classes = list(model.classes_)
        for i, idx in enumerate(np.where(test)[0]):
            rows.append({"system": system, "layer": layer, "filename": meta["filename"][idx],
                         "emotion": y[idx], "pred": pred[i],
                         "p_true": probs[i, classes.index(y[idx])],
                         "correct": int(pred[i] == y[idx])})

In [ ]:
# summary: mean and std across the 24 actors for every system/layer       # NEW
scores = pd.DataFrame(fold_scores)
summary = scores.groupby(["system", "layer"])[["macro_f1", "accuracy"]].agg(["mean", "std"])
print(summary.round(3))
summary.to_csv("results/classifier_summary.csv")
pd.DataFrame(rows).to_csv("results/predictions.csv", index=False)